In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

PROVINCE_COLORS = {
    'AJK':              '#4e9a8f',
    'Balochistan':      '#e07b39',
    'Gilgit-Baltistan': '#9b59b6',
    'ICT':              '#7f8c8d',
    'KPK':              '#2ca02c',
    'Punjab':           '#1f77b4',
    'Sindh':            '#d62728',
}

raw = pd.read_csv('../Data:/district_summary.csv')

# Apply sample-size filter
df = raw[(raw['n_government'] >= 100) & (raw['n_private'] >= 50)].copy()
df = df.dropna(subset=['arithmetic_gap', 'govt_mean_arithmetic'])

print(f'Total districts: {len(raw)}  →  after filter: {len(df)}')
df[['district', 'province', 'n_government', 'n_private', 'arithmetic_gap']].describe()

## Figure 1 — Top 30 Districts by Private-School Arithmetic Advantage

Top 30 districts by arithmetic gap (private − government), filtered to n_govt ≥ 100 and n_pvt ≥ 50.

In [ ]:
top30 = df.nlargest(30, 'arithmetic_gap').sort_values('arithmetic_gap')

bar_colors = [PROVINCE_COLORS[p] for p in top30['province']]
labels = [f"{r['district']}" for _, r in top30.iterrows()]

fig, ax = plt.subplots(figsize=(9, 11))

bars = ax.barh(labels, top30['arithmetic_gap'], color=bar_colors,
               edgecolor='white', linewidth=0.5, zorder=3)

# Value labels
for bar, val in zip(bars, top30['arithmetic_gap']):
    ax.text(
        val + 0.02, bar.get_y() + bar.get_height() / 2,
        f'{val:+.2f}', va='center', ha='left', fontsize=8.5, color='#333333'
    )

ax.axvline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=0)
ax.set_xlabel('Arithmetic gap  (private − government mean score)', labelpad=8)
ax.set_title('Top 30 Districts: Private-School Arithmetic Advantage\n'
             r'(n$_{govt}$ ≥ 100, n$_{pvt}$ ≥ 50)',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlim(left=0)

legend_handles = [
    mpatches.Patch(facecolor=color, label=prov)
    for prov, color in PROVINCE_COLORS.items()
    if prov in top30['province'].values
]
ax.legend(handles=legend_handles, title='Province', frameon=False,
          loc='lower right', fontsize=9, title_fontsize=9)

sns.despine(ax=ax, left=True)
ax.tick_params(axis='y', length=0)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()

fig.savefig(FIGURES / '03a_top30_districts_arithmetic_gap.png', bbox_inches='tight')
fig.savefig(FIGURES / '03a_top30_districts_arithmetic_gap.pdf', bbox_inches='tight')
plt.show()

## Figure 2 — Bottom 15 Districts: Government Schools Match or Beat Private Schools

Bottom 15 districts by arithmetic gap (most negative first), filtered to n_govt ≥ 100 and n_pvt ≥ 50.

In [ ]:
bot15 = df.nsmallest(15, 'arithmetic_gap').sort_values('arithmetic_gap', ascending=False)

bar_colors_b = [PROVINCE_COLORS[p] for p in bot15['province']]
labels_b = [r['district'] for _, r in bot15.iterrows()]

fig, ax = plt.subplots(figsize=(9, 6))

bars_b = ax.barh(labels_b, bot15['arithmetic_gap'], color=bar_colors_b,
                 edgecolor='white', linewidth=0.5, zorder=3)

# Value labels — place inside bar when bar extends far left, outside when short
for bar, val in zip(bars_b, bot15['arithmetic_gap']):
    offset = -0.02 if val < 0 else 0.02
    ha = 'right' if val < 0 else 'left'
    ax.text(
        val + offset, bar.get_y() + bar.get_height() / 2,
        f'{val:+.2f}', va='center', ha=ha, fontsize=8.5, color='#333333'
    )

ax.axvline(0, color='#444444', linewidth=1.0, linestyle='-', zorder=2)
ax.set_xlabel('Arithmetic gap  (private − government mean score)', labelpad=8)
ax.set_title('Bottom 15 Districts: Government Schools Match or Beat Private\n'
             r'(n$_{govt}$ ≥ 100, n$_{pvt}$ ≥ 50)',
             fontsize=12, fontweight='bold', pad=12)

legend_handles_b = [
    mpatches.Patch(facecolor=color, label=prov)
    for prov, color in PROVINCE_COLORS.items()
    if prov in bot15['province'].values
]
ax.legend(handles=legend_handles_b, title='Province', frameon=False,
          loc='lower left', fontsize=9, title_fontsize=9)

sns.despine(ax=ax, left=True)
ax.tick_params(axis='y', length=0)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()

fig.savefig(FIGURES / '03b_bottom15_districts_arithmetic_gap.png', bbox_inches='tight')
fig.savefig(FIGURES / '03b_bottom15_districts_arithmetic_gap.pdf', bbox_inches='tight')
plt.show()

## Table — 10 Districts with the Lowest Government School Arithmetic Scores

Ranked by absolute government school mean score (n_govt ≥ 100, n_pvt ≥ 50), independent of the private-government gap.

In [ ]:
worst10 = (
    df.nsmallest(10, 'govt_mean_arithmetic')
    [['district', 'province', 'govt_mean_arithmetic', 'pvt_mean_arithmetic',
      'arithmetic_gap', 'total_children_assessed']]
    .reset_index(drop=True)
)
worst10.index += 1  # 1-based rank

styled = worst10.style \
    .format({
        'govt_mean_arithmetic': '{:.2f}',
        'pvt_mean_arithmetic':  '{:.2f}',
        'arithmetic_gap':       '{:+.2f}',
        'total_children_assessed': '{:,.0f}',
    }) \
    .set_caption('10 Districts with Lowest Government School Arithmetic Score '
                 '(n_govt ≥ 100, n_pvt ≥ 50)') \
    .background_gradient(subset=['govt_mean_arithmetic'], cmap='Reds_r', vmin=2, vmax=5) \
    .background_gradient(subset=['arithmetic_gap'], cmap='RdYlGn_r', vmin=-0.5, vmax=1.5) \
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-weight', 'bold'), ('font-size', '11pt'), ('text-align', 'left')]
    }])

display(styled)

# Also print plain text for non-notebook use
print(worst10.rename(columns={
    'district': 'District',
    'province': 'Province',
    'govt_mean_arithmetic': 'Govt mean',
    'pvt_mean_arithmetic': 'Pvt mean',
    'arithmetic_gap': 'Gap',
    'total_children_assessed': 'N assessed',
}).to_string(float_format='{:.2f}'.format))